# Script to update metadata for any dataset

Check metadata for old sequences to see if anything has been added

Databases: GISAID, Andersen, NCBI Virus

In [ ]:
# Housekeeping


import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# Make sure you have the correct paths

# Dates
start_date = "11-01-2021"
end_date = "06-20-2025"
date_range = start_date + "--" + end_date
update_date = "06-24-2025"

references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"
# references = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/references/"
os.chdir(references)
states_ref = pd.read_csv("states_ref.csv")

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
# downloads_gisaid = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder for gisaid
# downloads_gisaid = "C:/Users/maksi/Downloads/"
downloads_gisaid = home + "GISAID/downloads/" + "2021-11-01--2025-04-14_North_America/" # Cats are North America
downloads_ncbi_virus = home + "NCBI_Virus/downloads/" # + date_range + "/"

genotype = "B3.13"
genotype_underscored = genotype.replace(".", "_")

originals = home + "Combinations/GISAID_Andersen_NCBI_Virus/" + date_range + "/" # "_cats/" # + "_" + genotype_underscored + "/"

complete = home + "Combinations/GISAID_Andersen_NCBI_Virus/" + date_range + "/updated_" + update_date + "/"

os.chdir(originals)

## Original Files

In [2]:
# Function to prepare dataframes
def fasta_df_og(file_name, state_ref):

    fasta = pd.DataFrame()
    headers = []
    isolate_ids = []
    isolate_names = []
    subtypes = []
    # segments = []
    collection_dates = []
    sequences = []
    host_types = []
    species = []
    identifiers = []
    genotypes = []
    with open(file_name) as f:
        lines = f.readlines()
        for num, line in enumerate(lines):
            # print(line)
            if line[0] == ">": # If it's a header
                if line[1:].strip() not in headers: # And the previous line is not a header we've seen before
                    header = line[1:].strip() # Remove the ">"
                    # print(header)
                    split_header = header.split("|")
                    if len(header.split("|")) > 6:
                        identifier = header.split("|")[0]
                        identifiers.append(identifier)
                    else:
                        identifiers.append("unknown")
                    split_first_header = split_header[-5].split("/")
                    # print(split_first_header)
                    # print(split_header)
                    headers.append(header) 
                    isolate_ids.append(split_first_header[3])
                    isolate_names.append(split_header[-5]) # We'll need to extract data from this too
                    # print(split_header[2].split("_")[-1])
                    subtypes.append(split_header[-4])  # Get only H5N1
                    genotypes.append(split_header[-1])
                    # segments.append(split_header[].split("_")[-1])
                    host_types.append(split_header[-2])
                    species.append(split_first_header[1])
                    # if split_header[4] == "2024-01-01":
                    #     collection_dates.append("2024") # No samples were collected 1/1/2024, these are all unknown 
                    # elif split_header[4] == "2025-01-01":
                    #     collection_dates.append("2025")
                    # else: 
                    collection_dates.append(split_header[-3].split("_")[-1])
                    if num < len(lines): # If we're not at the last line
                        # for i, l in enumerate(lines[num + 1:]):
                        i = num
                        sequence = ""
                        # print(lines[i])
                        # print(lines[i + 1])
                        while i < len(lines) - 1 and lines[i + 1][0] != ">": # While the next line is part of a sequence
                            sequence = sequence + lines[i + 1].strip()
                            i += 1
                        sequences.append(sequence) # Add next line to sequences
        f.close()

    # Create columns for data frame 
    fasta["Header"] = headers
    fasta["Isolate_Id"] = isolate_ids
    fasta["Isolate_Name"] = isolate_names
    fasta["Subtype"] = subtypes
    # fasta["Segment"] = segments
    # Geo_Location is more complicated
    fasta["Geo_Location"] = fasta["Header"].apply(lambda x: state_ref.loc[state_ref["Abbreviation"] == x.split("/")[2], 'Country'].iloc[0] + "-" + x.split("/")[2] if x.split("/")[2] in state_ref["Abbreviation"].values else state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Country'].iloc[0] + "-" + state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Abbreviation'].iloc[0] if x.split("/")[2].replace("_", " ") in state_ref["State"].values else x.split("/")[2].replace(": ", "-"))
    fasta["Date Collected"] = collection_dates
    fasta["Species"] = species
    fasta["Host_Type"] = host_types
    fasta["Genotype"] = genotypes
    fasta["Sequence"] = sequences
    if len(identifiers) == len(fasta):
        fasta["Identifier"] = identifiers
    
    return fasta

original_fasta_dfs = {}

for dirpath, dirs, files in os.walk(originals):
    for file in files:
        file_name = os.path.join(dirpath, file)
        if ".fasta" in file_name:
            fasta_file = fasta_df_og(file_name, states_ref)
            original_fasta_dfs[file_name] = fasta_file
            # print(fasta_file)
            # break 
    break 

## GISAID

In [ ]:
# Get data from GISAID

username = input("Username: ")
password = input("Password: ")
browser = input("Browser: ")
sleep_time = input("Seconds to sleep in between clicks: ")
continent = input("Continent(s) separated by commas: ")
start_date = dateutil.parser.parse(start_date).strftime("%Y-%m-%d") # Make sure date is in correct format
end_date = dateutil.parser.parse(end_date).strftime("%Y-%m-%d")

open_gisaid(username, password, browser, sleep_time, continent, start_date, end_date)

In [4]:
# Get downloaded GISAID data

all_metadata_files = []
all_fasta_files = []

# Grab files
for dirpath, dirs, files in os.walk(downloads_gisaid):
    for file in files:
        file_name = os.path.join(dirpath, file)
        # print(file_name)
        if ".xls" in file_name:
            metadata = pd.read_excel(file_name, engine="xlrd")
            all_metadata_files.append(metadata)
        if ".fasta" in file_name:
            fasta_file = fasta_df(file_name, states_ref) # Convert fasta file to dataframe
            all_fasta_files.append(fasta_file)

In [5]:
# Search for dates and states based on isolate

for key in original_fasta_dfs:
    og_df = original_fasta_dfs[key]
    print(og_df)
    og_df["Unknown_States"] = og_df["Geo_Location"].apply(lambda x: 1 if x == "USA" else 0)
    og_df["Unknown_Dates"] = og_df["Date Collected"].apply(lambda x: 1 if x == "2022-01-01" or x == "2023-01-01" or x == "2024-01-01" or x == "2025-01-01" else 0)
    og_df["Update_Needed"] = og_df["Unknown_States"] + og_df["Unknown_Dates"]
    # og_df["Identifier"] = ""
    og_df_update_needed = og_df[og_df["Update_Needed"] > 0]
    for isolate in og_df_update_needed["Isolate_Id"].values:
        # print(isolate)
        for new_df in all_fasta_files:
            if isolate in new_df["Isolate_Id"].values:
                # print(isolate)
                og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Geo_Location"] = new_df.loc[new_df[new_df["Isolate_Id"] == isolate].index[0], "Geo_Location"]
                og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Date Collected"] = new_df.loc[new_df[new_df["Isolate_Id"] == isolate].index[0], "Date Collected"]
                og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Identifier"] = new_df.loc[new_df[new_df["Isolate_Id"] == isolate].index[0], "Identifier"] # .split("|")[0]

    og_df_update_needed = og_df_update_needed.drop_duplicates(subset="Header")
    print(og_df_update_needed)

    # new_df = og_df.merge(og_df_update_needed, how="left")
    new_df = og_df.set_index('Header')
    new_df.update(og_df_update_needed.set_index('Header'))
    new_df = new_df.reset_index()
    # new_df = pd.concat([og_df_update_needed, og_df]).drop_duplicates(['Isolate_Id'], keep="last")
    print(new_df)
    # break 

    original_fasta_dfs[key] = new_df

    

                                                Header  \
0    A/gull/Oregon/23-000775-001/2023|H5N1|2023-01-...   
1    A/gull/Oregon/23-000764-001/2023|H5N1|2023-01-...   
2    A/gull/California/23-004370-004/2023|H5N1|2023...   
3    A/gull/California/23-004370-003/2023|H5N1|2023...   
4    A/raven/California/23-001056-002/2022|H5N1|202...   
..                                                 ...   
247  A/branta canadensis/USA: IL/25-000616-003-orig...   
248  A/branta canadensis/USA: IL/25-000616-006-orig...   
249  A/branta canadensis/USA: IL/25-000616-008-orig...   
250  A/gallus gallus/USA: ND/24-037793-001-original...   
251  A/anatidae/USA: ND/24-037793-003-original/2024...   

                 Isolate_Id  \
0             23-000775-001   
1             23-000764-001   
2             23-004370-004   
3             23-004370-003   
4             23-001056-002   
..                      ...   
247  25-000616-003-original   
248  25-000616-006-original   
249  25-000616-008-origi

## Andersen

In [6]:
# Read metadata

os.chdir(home)
metadata_normalized = pd.read_csv("metadata_normalized.tsv", delimiter="\t") # Collection dates

metadata_folder = home + "Andersen/avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv") # Everything else

print(len(metadata))

# Merge with metadata_normalized
metadata = metadata.merge(metadata_normalized, how="outer")
print(metadata.columns)

# # Find only >= last date using Release Date from metadata 
# metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
# metadata = metadata[metadata["ReleaseDate"] >= dateutil.parser.parse(start_date).strftime("%Y-%m-%d")]
# # Find only <= update date using Release Date from metadata
# metadata = metadata[metadata["ReleaseDate"] <= dateutil.parser.parse(end_date).strftime("%Y-%m-%d")]

print(len(metadata)) 
display(metadata)

9697
Index(['Run', 'Assay Type', 'AvgSpotLen', 'Bases', 'BioProject', 'BioSample',
       'BioSampleModel', 'Bytes', 'Center Name', 'Collection_Date', 'Consent',
       'DATASTORE filetype', 'DATASTORE provider', 'DATASTORE region',
       'Experiment', 'geo_loc_name_country', 'geo_loc_name_country_continent',
       'geo_loc_name', 'Host', 'Instrument', 'isolate', 'Library Name',
       'LibraryLayout', 'LibrarySelection', 'LibrarySource', 'Organism',
       'Platform', 'ReleaseDate', 'create_date', 'version', 'Sample Name',
       'SRA Study', 'serotype', 'isolation_source', 'BioSample Accession',
       'is_retracted', 'retraction_detection_date_utc'],
      dtype='object')
19050


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,ReleaseDate,create_date,version,Sample Name,SRA Study,serotype,isolation_source,BioSample Accession,is_retracted,retraction_detection_date_utc
0,SRR24839058,AMPLICON,230.77,32942716,PRJNA980729,SAMN35647642,Viral,18373327,United States Department of Agriculture,missing,...,2023-06-30 00:46:13,2023-06-07 02:01:47,1,22-005893-001,SRP441379,H5N1,",",SRS17903639,False,NaN
1,SRR24839058,AMPLICON,230.77,32942716,PRJNA980729,SAMN35647642,Viral,18373327,United States Department of Agriculture,missing,...,2023-06-30 00:46:13,2023-06-07 02:01:47,1,22-005893-001,SRP441379,H5N1,"Swab, Tracheal",SRS17903639,False,NaN
2,SRR24839059,AMPLICON,207.54,53993591,PRJNA980729,SAMN35647620,Viral,29609119,United States Department of Agriculture,2022-02-08,...,2023-06-30 00:46:13,2023-06-07 02:01:22,1,AH0210318,SRP441379,H5N1,",/",SRS17903636,False,NaN
3,SRR24839059,AMPLICON,207.54,53993591,PRJNA980729,SAMN35647620,Viral,29609119,United States Department of Agriculture,2022-02-08,...,2023-06-30 00:46:13,2023-06-07 02:01:22,1,AH0210318,SRP441379,H5N1,"Swab Pool, Cloacal/Oropharyngeal",SRS17903636,False,NaN
4,SRR24839060,AMPLICON,201.66,21703662,PRJNA980729,SAMN35647621,Viral,10716272,United States Department of Agriculture,2022-02-17,...,2023-06-30 00:46:13,2023-06-07 02:01:28,1,22-005158-001,SRP441379,H5N1,NaN,SRS17903631,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19045,SRR33830446,WGS,148.77,95232771,PRJNA1102327,SAMN48895426,Viral,37952018,USDA-NVSL,2025,...,2025-06-06 00:57:17,2025-06-04 14:24:56,1,25-015677-004,SRP503016,NaN,"MILK, BULK TANK",SRS25266464,False,NaN
19046,SRR33830447,WGS,148.43,106145246,PRJNA1102327,SAMN48895425,Viral,42182759,USDA-NVSL,2025,...,2025-06-06 00:57:17,2025-06-04 14:25:01,1,25-015677-003,SRP503016,NaN,"MILK, BULK TANK",SRS25266463,False,NaN
19047,SRR33830448,WGS,148.30,110812426,PRJNA1102327,SAMN48895424,Viral,43738503,USDA-NVSL,2025,...,2025-06-06 00:57:17,2025-06-04 14:24:55,1,25-015677-002,SRP503016,NaN,"MILK, BULK TANK",SRS25266462,False,NaN
19048,SRR33830449,WGS,131.52,67358891,PRJNA1102327,SAMN48895415,Viral,26336550,USDA-NVSL,2024,...,2025-06-06 00:57:16,2025-06-04 14:24:53,1,24-036379-001-tile,SRP503016,NaN,"MILK, BULK TANK",SRS25266461,False,NaN


In [7]:
# Collapse dataset to only sequences in original dataset AND without dates OR states

for key in original_fasta_dfs:
    og_df = original_fasta_dfs[key]
    # print(og_df)
    og_df["Unknown_States"] = og_df["Geo_Location"].apply(lambda x: 1 if x == "USA" else 0)
    og_df["Unknown_Dates"] = og_df["Date Collected"].apply(lambda x: 1 if x == "2023" or x == "2024" or x == "2025" else 0)
    og_df["Update_Needed"] = og_df["Unknown_States"] + og_df["Unknown_Dates"]
    # og_df["Identifier"] = ""
    og_df_update_needed = og_df[og_df["Update_Needed"] > 0]
    for isolate in og_df_update_needed["Isolate_Id"].values:
        # print(isolate)
        # for new_df in metadata:
        if isolate in metadata["isolate"].values:
            # print(isolate)
            og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Geo_Location"] = metadata.loc[metadata[metadata["isolate"] == isolate].index[0], "geo_loc_name"]
            og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Date Collected"] = metadata[metadata["isolate"] == isolate]["BioSample"].apply(lambda x: search_collection_date(x, metadata)).values[0]
            og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Identifier"] = metadata.loc[metadata[metadata["isolate"] == isolate].index[0], "Run"]

    print(og_df_update_needed)

    # new_df = og_df.merge(og_df_update_needed, how="left")
    # new_df = pd.concat([og_df_update_needed, og_df]).drop_duplicates('Isolate_Id', keep="first")
    new_df = og_df.set_index('Header')
    new_df.update(og_df_update_needed.set_index('Header'))
    new_df = new_df.reset_index()

    print(new_df)
    # break 

    original_fasta_dfs[key] = new_df

                                                Header  \
87   A/bald_eagle/USA/004135-002/2025|H5N1|2025|avi...   
94    A/chicken/USA/034633-001/2024|H5N1|2024|avian|A3   
95    A/chicken/USA/034633-002/2024|H5N1|2024|avian|A3   
96    A/chicken/USA/000536-001/2025|H5N1|2025|avian|A3   
97    A/chicken/USA/000537-002/2025|H5N1|2025|avian|A3   
98    A/chicken/USA/000536-002/2025|H5N1|2025|avian|A3   
99    A/chicken/USA/000537-001/2025|H5N1|2025|avian|A3   
100   A/chicken/USA/038441-002/2024|H5N1|2024|avian|A3   
102  A/bald_eagle/USA/000778-001/2025|H5N1|2025|avi...   
103  A/red-tailed_hawk/USA/001425-001/2025|H5N1|202...   
105   A/chicken/USA/000282-001/2025|H5N1|2025|avian|A3   
109  A/fox/USA/000628-001/2025|H5N1|2025|other_mamm...   
112   A/grackle/USA/009017-003/2025|H5N1|2025|avian|A3   
119  A/bald_eagle/USA/005228-001/2025|H5N1|2025|avi...   
125  A/bald_eagle/USA/008364-001/2025|H5N1|2025|avi...   
126  A/bald_eagle/USA/006076-002/2025|H5N1|2025|avi...   
127   A/mallar

## NCBI Virus

In [8]:
os.chdir(downloads_ncbi_virus + date_range + "/")

# Read metadata
ncbi_metadata = pd.read_csv("sequences.csv")

# # Integrate genotypes
# os.chdir(downloads_ncbi_virus + "11-01-2021--04-14-2025/")
# apr_output = pd.read_csv("output.tsv", delimiter="\t")

# os.chdir(downloads_ncbi_virus + "04-14-2025--05-14-2025/")
# may_output = pd.read_csv("output.tsv", delimiter="\t")

print(ncbi_metadata)

      Accession      Organism_Name GenBank_RefSeq         Assembly  \
0      PV570239  Influenza A virus        GenBank              NaN   
1      PV571926  Influenza A virus        GenBank  GCA_049972485.1   
2      PV571927  Influenza A virus        GenBank  GCA_049972485.1   
3      PV571928  Influenza A virus        GenBank  GCA_049972485.1   
4      PV571929  Influenza A virus        GenBank  GCA_049972485.1   
...         ...                ...            ...              ...   
78929  ON759332  Influenza A virus        GenBank              NaN   
78930  ON759333  Influenza A virus        GenBank              NaN   
78931  ON759334  Influenza A virus        GenBank              NaN   
78932  ON759335  Influenza A virus        GenBank              NaN   
78933  ON759336  Influenza A virus        GenBank              NaN   

      SRA_Accession                                         Submitters  \
0               NaN  Iervolino,M., Guenther,A., Begeman,L., Aguado,...   
1       SRR

In [9]:
# Search for dates and states based on isolate

for key in original_fasta_dfs:
    og_df = original_fasta_dfs[key]
    # print(og_df)
    og_df["Unknown_States"] = og_df["Geo_Location"].apply(lambda x: 1 if x == "USA" else 0)
    og_df["Unknown_Dates"] = og_df["Date Collected"].apply(lambda x: 1 if x == "2023" or x == "2024" or x == "2025" else 0)

    # og_df["Unknown_Dates"] = og_df["Date Collected"].apply(dateutil.parser.parse).apply(lambda x: 1 if x == dateutil.parser.parse("2023-01-01") or x == dateutil.parser.parse("2024-01-01") or x == dateutil.parser.parse("2025-01-01") else 0)
    og_df["Update_Needed"] = og_df["Unknown_States"] + og_df["Unknown_Dates"]
    # og_df["Identifier"] = ""
    og_df_update_needed = og_df[og_df["Update_Needed"] > 0]
    # print(og_df_update_needed)
    for isolate in og_df_update_needed["Isolate_Id"].values:
        # print(isolate)
        # for new_df in ncbi_metadata:
            # print(new_df)
        if isolate in ncbi_metadata["Isolate"].values:
            print(isolate)
            og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Geo_Location"] = ncbi_metadata.loc[ncbi_metadata[ncbi_metadata["Isolate"] == isolate].index[0], "Geo_Location"] # .replace(": ", "-")
            og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Date Collected"] = ncbi_metadata.loc[ncbi_metadata[ncbi_metadata["Isolate"] == isolate].index[0], "Collection_Date"]
            og_df_update_needed.at[og_df_update_needed[og_df_update_needed["Isolate_Id"] == isolate].index[0], "Identifier"] = ncbi_metadata.loc[ncbi_metadata[ncbi_metadata["Isolate"] == isolate].index[0], "SRA_Accession"]

    # new_df = og_df.merge(og_df_update_needed, how="left")
    # new_df = pd.concat([og_df_update_needed, og_df]).drop_duplicates('Isolate_Id', keep="first")

    new_df = og_df.set_index('Header')
    new_df.update(og_df_update_needed.set_index('Header'))
    new_df = new_df.reset_index()

    print(new_df)
    # break 

    original_fasta_dfs[key] = new_df

25-000616-003-original
25-000616-006-original
25-000616-008-original
                                                Header  \
0    A/gull/Oregon/23-000775-001/2023|H5N1|2023-01-...   
1    A/gull/Oregon/23-000764-001/2023|H5N1|2023-01-...   
2    A/gull/California/23-004370-004/2023|H5N1|2023...   
3    A/gull/California/23-004370-003/2023|H5N1|2023...   
4    A/raven/California/23-001056-002/2022|H5N1|202...   
..                                                 ...   
247  A/branta canadensis/USA: IL/25-000616-003-orig...   
248  A/branta canadensis/USA: IL/25-000616-006-orig...   
249  A/branta canadensis/USA: IL/25-000616-008-orig...   
250  A/gallus gallus/USA: ND/24-037793-001-original...   
251  A/anatidae/USA: ND/24-037793-003-original/2024...   

                 Isolate_Id  \
0             23-000775-001   
1             23-000764-001   
2             23-004370-004   
3             23-004370-003   
4             23-001056-002   
..                      ...   
247  25-000616-00

# Put it all together

In [ ]:

for key in original_fasta_dfs:
    df = original_fasta_dfs[key]
    df["Identifier"] = df["Identifier"].apply(lambda x: "unknown" if x != x else x) # Put "unknown" if NaN
    df = df[df["Genotype"] == genotype] # Make sure we only have the genotype we want
    df["full_header"] = ">" + df["Identifier"] + "|" + df["Isolate_Name"] + "|" + df["Subtype"] + "|" + df["Geo_Location"] + "|" + df["Date Collected"] + "|" + df["Host_Type"] + "|" + df["Genotype"] + "\n"
    df["sequence"] = df["Sequence"]

    df_to_fasta(df, key.split("/")[-1][:-6] + date_range + "_updated_" + update_date + ".fasta", complete)